# Lab 2 Solving a 3×3 Sudoku using Grover's Algorithm <a id="sudoku"></a>

We will now solve a simple problem using Grover's algorithm, for which we do not necessarily know the solution beforehand. Our problem is a 3×3 Sudoku-style puzzle. Since a 3×3 puzzle cannot be binary if no value is allowed to repeat within a row or column, we will use three possible values: 0, 1, and 2.

The puzzle has two rules:

- No row may contain the same value twice.
- No column may contain the same value twice.

Equivalently, each row and each column must contain the values 0, 1, and 2 exactly once. This is a small Latin-square version of Sudoku.

If we assign each square in our Sudoku to a variable like so:

| v0 | v1 | v2 |
| -- | -- | -- |
| v3 | v4 | v5 |
| v6 | v7 | v8 |

we want our circuit to output an assignment of values to these variables that satisfies all the row and column constraints.

Note that, while this approach of using Grover's algorithm to solve this problem is not practical for such a small puzzle, the purpose of this example is to demonstrate the conversion of classical decision problems into oracles for Grover's algorithm.

### Turning the Problem into a Circuit

We want to create an oracle that will help us solve this problem. We start by creating a reversible classical circuit that identifies a correct solution. In other words, we need a classical function implemented on a quantum circuit that checks whether the state of the variable registers represents a valid solution.

Since we need to check across all three rows and down all three columns, there are 18 pairwise conditions we need to check:

```
# Check across the first row
v0 ≠ v1
v0 ≠ v2
v1 ≠ v2

# Check across the second row
v3 ≠ v4
v3 ≠ v5
v4 ≠ v5

# Check across the third row
v6 ≠ v7
v6 ≠ v8
v7 ≠ v8

# Check down the first column
v0 ≠ v3
v0 ≠ v6
v3 ≠ v6

# Check down the second column
v1 ≠ v4
v1 ≠ v7
v4 ≠ v7

# Check down the third column
v2 ≠ v5
v2 ≠ v8
v5 ≠ v8
```

Because each variable can take three values, each variable may be encoded using two qubits, for example 00, 01, and 10 for the values 0, 1, and 2. Therefore, the oracle should check two things:

```
1. Every variable encodes a valid value: 0, 1, or 2.
2. All row and column inequality constraints are satisfied.
```



1. Create a reversible classical circuit that identifies a correct 3×3 solution.
2. Turn this circuit into an oracle.
3. Use Grover's algorithm to search for a valid assignment.

### Optional starting grid

To make the problem more Sudoku-like, you may fix one or more values before running Grover's algorithm. For example:



| 0 |   |   |
| - | - | - |
|   | 1 |   |
|   |   | 2 |

This adds the following constraints:

```
v0 = 0
v4 = 1
v8 = 2
```

The oracle should mark only the states that satisfy both the fixed-value constraints and the row/column constraints.

### This notebook solves the 3×3 Sudoku-style Grover lab with a compact row-permutation encoding. It is designed to use fewer two-qubit gates and smaller two-qubit depth than a direct two-qubits-per-cell encoding.

- V-chain multi-controlled gates with clean ancillas to reduce circuit depth;
- selectable fixed-value difficulty: `easy`, `medium`, or `hard`;
- hardware-aware transpilation using the selected backend target/coupling map;
- optional real-backend execution template with error suppression and mitigation;
- simulation path kept separate from hardware execution, avoiding Aer target/basis errors.

> The `hard` option follows the requested requirement of three fixed values, but uses a different clue pattern from `easy`.


In [1]:
# Install packages in a fresh environment if needed.
# %pip install -q qiskit qiskit-aer qiskit-ibm-runtime 
# %pip install "iqm-client[qiskit]" qiskit-ionq

import itertools
import random
import numpy as np
import math
import pandas as pd
from collections import Counter

from qiskit import QuantumCircuit, transpile
from qiskit.converters import circuit_to_dag
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager

# QiskitRuntimeService cannot work if one has to view IQM metrics. IQM metrics only works with qiskit 1X versions. 
#from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler 

try:
    from qiskit_aer import AerSimulator
    AER_AVAILABLE = True
except Exception:
    AER_AVAILABLE = False

print('Aer available:', AER_AVAILABLE)

Aer available: True


### Difficulty selection
Choose one of: easy or medium or hard or empty

In [2]:
# Difficulty selection
# Choose one of: "easy", "medium", "hard", "empty"
# DIFFICULTY = input('Enter the difficulty level (choose - easy or medium or hard or empty: ')

DIFFICULTY = input('Enter the difficulty level (choose - easy or medium or hard or empty): ').strip().lower()

VALUES = (0, 1, 2)
PERMS = list(itertools.permutations(VALUES))

# Fixed values are stored as {(row, column): value}; all indices are 0-based.
# Hard intentionally has no fixed values, so it produces the largest compact search space.
PUZZLES = {
    "easy": dict(zip([(0, 0), (1, 1), (2, 2)],random.sample([0, 1, 2], k=3))),
    "medium": dict(zip([(0, 0), (1, 1)],random.sample([0, 1, 2], k=3))),
    "hard": dict(zip([(0, 0)],random.sample([0, 1, 2], k=3))),
    "empty": {}
}

if DIFFICULTY not in PUZZLES:
    raise ValueError(f'DIFFICULTY must be one of {list(PUZZLES)}')

FIXED_VALUES = PUZZLES[DIFFICULTY]

def print_grid(fixed_values):
    for r in range(3):
        row = []
        for c in range(3):
            row.append(str(fixed_values.get((r, c), '.')))
        print(' '.join(row))

print('Selected difficulty:', DIFFICULTY)
print('Fixed starting grid:')
print_grid(FIXED_VALUES)


Selected difficulty: medium
Fixed starting grid:
2 . .
. 0 .
. . .


#### Convert fixed clues into legal row permutations.

#### Each row is always a permutation of 0,1,2, so row constraints are removed before circuit construction. The quantum oracle only needs to check columns.

In [3]:
# Convert fixed clues into legal row permutations.
# Each row is always a permutation of 0,1,2, so row constraints are removed before
# circuit construction. The quantum oracle only needs to check columns.

def row_choices_for_fixed_values(fixed_values):
    choices = []
    for r in range(3):
        row_choices = []
        for perm in PERMS:
            ok = True
            for (rr, cc), value in fixed_values.items():
                if rr == r and perm[cc] != value:
                    ok = False
                    break
            if ok:
                row_choices.append(perm)
        if not row_choices:
            raise ValueError(f'No legal permutations remain for row {r}.')
        choices.append(row_choices)
    return choices

ROW_CHOICES = row_choices_for_fixed_values(FIXED_VALUES)

for r, choices in enumerate(ROW_CHOICES):
    print(f'Row {r}: {len(choices)} legal choices -> {choices}')

Row 0: 2 legal choices -> [(2, 0, 1), (2, 1, 0)]
Row 1: 2 legal choices -> [(1, 0, 2), (2, 0, 1)]
Row 2: 6 legal choices -> [(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]


In [4]:
# Compact binary encoding for row choices

def bits_needed(num_choices):
    return max(1, math.ceil(math.log2(num_choices)))

ROW_WIDTHS = [bits_needed(len(c)) for c in ROW_CHOICES]
OFFSETS = [0]
for w in ROW_WIDTHS[:-1]:
    OFFSETS.append(OFFSETS[-1] + w)

NUM_QUBITS = sum(ROW_WIDTHS)
SEARCH_SPACE_SIZE = 2 ** NUM_QUBITS

def bits_to_int(bits):
    return sum(bit << i for i, bit in enumerate(bits))

def decode_assignment(bitstring):
    """Decode little-endian bitstring in Qiskit qubit order q0,q1,..."""
    rows = []
    for offset, width, choices in zip(OFFSETS, ROW_WIDTHS, ROW_CHOICES):
        idx = bits_to_int(bitstring[offset:offset + width])
        if idx >= len(choices):
            return None
        rows.append(choices[idx])
    return rows

def is_valid_sudoku_rows(rows):
    if rows is None:
        return False
    return all(set(rows[r][c] for r in range(3)) == set(VALUES) for c in range(3))

MARKED_BITSTRINGS = []
for bits in itertools.product([0, 1], repeat=NUM_QUBITS):
    rows = decode_assignment(bits)
    if is_valid_sudoku_rows(rows):
        MARKED_BITSTRINGS.append(bits)

print(f'Search qubits: {NUM_QUBITS}')
print(f'Row widths: {ROW_WIDTHS}')
print(f'Search space size: {SEARCH_SPACE_SIZE}')
print(f'Marked states: {len(MARKED_BITSTRINGS)}')
print('Marked bitstrings in q0,q1,... order:', MARKED_BITSTRINGS)

for bits in MARKED_BITSTRINGS:
    print('\nSolution from bitstring', bits)
    for row in decode_assignment(bits):
        print(row)

Search qubits: 5
Row widths: [1, 1, 3]
Search space size: 32
Marked states: 1
Marked bitstrings in q0,q1,... order: [(1, 0, 1, 0, 0)]

Solution from bitstring (1, 0, 1, 0, 0)
(2, 1, 0)
(1, 0, 2)
(0, 2, 1)


#### Oracle and diffuser with optional V-chain MCX

#### V-chain MCX usually has much lower depth than no-ancilla MCX. The trade-off is that it needs clean helper ancillas. For an n-qubit phase flip, the MCX has n-1 controls and therefore needs max(0, n-3) clean ancillas in v-chain mode.

#### The ancillas are required based on the fixed clues. For e.g easy does'nt need an V-chain MCX. 

In [5]:
USE_V_CHAIN = True
MCX_MODE = 'v-chain' if USE_V_CHAIN else 'noancilla'
NUM_ANCILLAS = max(0, NUM_QUBITS - 3) if USE_V_CHAIN else 0

print('MCX mode:', MCX_MODE)
print('Clean ancillas allocated:', NUM_ANCILLAS)

def _mcx(qc, controls, target, ancillas=None):
    """Version-tolerant MCX wrapper."""
    ancillas = [] if ancillas is None else list(ancillas)
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    elif len(controls) == 2:
        qc.ccx(controls[0], controls[1], target)
    elif USE_V_CHAIN:
        required = max(0, len(controls) - 2)
        if len(ancillas) < required:
            raise ValueError(f'V-chain MCX needs {required} clean ancillas for {len(controls)} controls.')
        # Qiskit versions differ slightly in mcx argument names, so try both forms.
        try:
            qc.mcx(controls, target, ancilla_qubits=ancillas[:required], mode='v-chain')
        except TypeError:
            qc.mcx(controls, target, ancillas[:required], mode='v-chain')
    else:
        qc.mcx(controls, target, mode='noancilla')

def apply_phase_on_bitstring(qc, qubits, bitstring, ancillas=None):
    """Apply a -1 phase to exactly one computational-basis state."""
    if len(qubits) != len(bitstring):
        raise ValueError('qubits and bitstring must have the same length')

    for q, bit in zip(qubits, bitstring):
        if bit == 0:
            qc.x(q)

    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        _mcx(qc, qubits[:-1], qubits[-1], ancillas=ancillas)
        qc.h(qubits[-1])

    for q, bit in zip(qubits, bitstring):
        if bit == 0:
            qc.x(q)

def apply_oracle(qc, qubits, marked_bitstrings, ancillas=None):
    for bits in marked_bitstrings:
        apply_phase_on_bitstring(qc, qubits, bits, ancillas=ancillas)

def apply_diffuser(qc, qubits, ancillas=None):
    qc.h(qubits)
    qc.x(qubits)
    apply_phase_on_bitstring(qc, qubits, tuple(1 for _ in qubits), ancillas=ancillas)
    qc.x(qubits)
    qc.h(qubits)

def grover_iterations(num_states, num_marked):
    if num_marked <= 0:
        raise ValueError('The oracle has no marked states.')
    return max(1, round((math.pi / 4) * math.sqrt(num_states / num_marked)))

ITERATIONS = grover_iterations(SEARCH_SPACE_SIZE, len(MARKED_BITSTRINGS))
print('Grover iterations:', ITERATIONS)


MCX mode: v-chain
Clean ancillas allocated: 2
Grover iterations: 4


#### Build the compact Grover circuit
#### Data qubits are measured; V-chain ancillas are left unmeasured and returned clean.

In [6]:
data_q = list(range(NUM_QUBITS))
ancilla_q = list(range(NUM_QUBITS, NUM_QUBITS + NUM_ANCILLAS))

grover = QuantumCircuit(NUM_QUBITS + NUM_ANCILLAS, NUM_QUBITS, name=f'compact_grover_{DIFFICULTY}_{MCX_MODE}')

grover.h(data_q)
for _ in range(ITERATIONS):
    apply_oracle(grover, data_q, MARKED_BITSTRINGS, ancillas=ancilla_q)
    apply_diffuser(grover, data_q, ancillas=ancilla_q)

grover.measure(data_q, range(NUM_QUBITS))

print(grover.draw(fold=-1))


     ┌───┐                ┌───┐┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐                     ┌───┐┌───┐     ┌─┐            
q_0: ┤ H ├────────────■───┤ H ├┤ X ├─────────────────■───┤ X ├┤ H ├─────────────────■───┤ H ├┤ X ├─────────────────■───┤ X ├┤ H ├─────────────────■───┤ H ├┤ X ├─────────────────■───┤ X ├┤ H ├─────────────────■───┤ H ├┤ X ├─────────────────■───┤ X ├┤ H ├─────┤M├────────────
     ├───┤┌───┐       │   ├───┤├───┤┌───┐            │   ├───┤├───┤┌───┐            │   ├───┤├───┤┌───┐            │   ├───┤├───┤┌───┐            │   ├───┤├───┤┌───┐            │   ├───┤├───┤┌───┐            │   ├───┤├───┤┌───┐            │   ├───┤├───┤     └╥┘┌─┐         
q_1: ┤ H ├┤ X ├───────■───┤ X ├┤ H ├┤ X ├────────────■───┤ X ├┤ H ├┤ X ├────────────■───┤ X ├┤ H ├┤ X ├────────────■───┤ X ├┤ H ├┤ X ├────────────■───┤ X ├┤ H ├┤ X ├────────────■

#### Metrics helper funnctions focused on total gate depth and two-qubit depth

In [7]:
TWO_QUBIT_NAMES = {
    'cx', 'cz', 'ecr', 'swap', 'iswap', 'rxx', 'ryy', 'rzz', 'rzx', 'xx_plus_yy', 'mcx', 'ccx'
}

def two_qubit_gate_count(circuit):
    total = 0
    for inst in circuit.data:
        if len(inst.qubits) == 2:
            total += 1
    return total

def two_qubit_depth(circuit):
    dag = circuit_to_dag(circuit)
    depth = 0
    for layer in dag.layers():
        if any(len(node.qargs) >= 2 for node in layer['graph'].op_nodes()):
            depth += 1
    return depth

def metrics(circuit, label):
    no_meas = circuit.remove_final_measurements(inplace=False)
    return {
        'circuit': label,
        'qubits': circuit.num_qubits,
        'total_depth_no_measure': no_meas.depth(),
        'two_qubit_depth': two_qubit_depth(no_meas),
        #'two_qubit_gate_count': two_qubit_gate_count(no_meas),
        'ops_no_measure': dict(no_meas.count_ops()),
    }

#metrics(grover, 'logical compact circuit')

## 1. Noiseless Aer local simulation

Simulate a simulator-transpiled circuit, not the hardware-transpiled circuit.


In [8]:
basis_gates=['rz', 'sx', 'x', 'cx', 'measure']

if AER_AVAILABLE:
    simulator = AerSimulator()
    simulation_circuit = transpile(grover, optimization_level=3,
                                   basis_gates=basis_gates)
    result = simulator.run(simulation_circuit, shots=2048).result()
    counts = result.get_counts()
    print(simulation_circuit.draw(fold=-1))
    print(counts)

    best_key = max(counts, key=counts.get)
    best_bits = tuple(int(b) for b in best_key[::-1])
    print('\nMost likely key:', best_key)
    print('As q0,q1,... bits:', best_bits)
    print('Decoded solution:')
    for row in decode_assignment(best_bits):
        print(row)
else:
    print('qiskit-aer is not installed, so local simulation was skipped.')

     ┌─────────┐ ┌────┐                                                                                                                                                                                                                                                                                                                                                                       ┌────┐   ┌───────┐                                                                                                                                                                                                                                                                                                                                                                                                                                      ┌────┐   ┌────────┐                                                                                                                                                               

#### Transpiled Circuit Metrics (Gate depths and 2Q depths)

In [9]:
print('\n Metrics:')
metrics(simulation_circuit, 'transpiled_aer')


 Metrics:


{'circuit': 'transpiled_aer',
 'qubits': 7,
 'total_depth_no_measure': 383,
 'two_qubit_depth': 137,
 'ops_no_measure': {'rz': 245, 'cx': 137, 'sx': 116}}

## Real-backend error suppression and mitigation

The next cells are optional and are meant for IBM Quantum real backends. `RUN_ON_REAL_BACKEND = False` is default. True to make it run on real backend

Techniques included:

1. **Hardware-aware transpilation** to the real backend target with high optimization.
2. **Error suppression** through Runtime options such as dynamical decoupling and Pauli/gate twirling when supported by your installed `qiskit-ibm-runtime` version.
3. **Measurement/readout mitigation** using calibration circuits and an assignment-matrix correction.

#### Tip: Run on real backends so that the error supression and mitigation techniques have effects on the final measurements

In [10]:
# Optional real-backend run with Runtime Sampler and suppression options

RUN_ON_REAL_BACKEND = True
SHOTS = 2048

real_counts = None

if RUN_ON_REAL_BACKEND:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    service = QiskitRuntimeService()  # requires a saved IBM Quantum account
    backend = service.least_busy(simulator=False, operational=True)
    print('Selected backend:', backend.name)
    fake_backend = AerSimulator.from_backend(backend)
    
    real_circuit = transpile(
        grover,
        backend=fake_backend,
        optimization_level=3,
        )

    sampler = Sampler(mode = fake_backend)

    # Error suppression: dynamical decoupling during idle time.
    try:
        sampler.options.dynamical_decoupling.enable = True
        sampler.options.dynamical_decoupling.sequence_type = 'XpXm'
        print('Enabled dynamical decoupling.')
    except Exception as exc:
        print('Dynamical decoupling option not available in this Runtime version:', exc)

    # Error suppression: Pauli/gate twirling converts coherent errors into stochastic errors.
    try:
        sampler.options.twirling.enable_gates = True
        sampler.options.twirling.num_randomizations = 'auto'
        sampler.options.twirling.enable_measure = True
        print('Enabled gate twirling.')
    except Exception as exc:
        print('Twirling option not available in this Runtime version:', exc)

    job = sampler.run([real_circuit], shots=SHOTS)
    pub_result = job.result()[0]

    # Most current SamplerV2 result objects expose bit-array counts through data.<classical_register>.get_counts().
    creg_name = grover.cregs[0].name
    real_counts = getattr(pub_result.data, creg_name).get_counts()
    print(real_counts)
else:
    print('Set RUN_ON_REAL_BACKEND = False to submit to a real backend.')


ImportError: cannot import name 'MeasLevel' from 'qiskit.result' (e:\UA_Little_Rock\Labs\.venv\Lib\site-packages\qiskit\result\__init__.py)

In [ ]:
import mthree

meas_map = mthree.utils.final_measurement_mapping(real_circuit)
mit = mthree.M3Mitigation(fake_backend)

mit.cals_from_system(meas_map)

m3_quasi = mit.apply_correction(real_counts, meas_map)

def convert_bin_list(dist, shots=10000):
    # Step 1: convert to (non-negative) counts
    counts = {
        k: (int(round(float(v) * shots)) if float(v) >= 0 else 0)
        for k, v in dist.items()
    }
    # Step 2: normalize
    total_counts = sum(counts.values())
    return {k: round(v / total_counts, int(np.log10(shots))) for k, v in counts.items()}

final_distribution = convert_bin_list(dict(m3_quasi), shots=SHOTS)
final_distribution = dict(
    sorted(final_distribution.items(), key=lambda item: item[1], reverse=True)
)
print("Results (M3 mitigation only + error suppression):")
print(final_distribution)
print()

ImportError: cannot import name 'MeasLevel' from 'qiskit.result' (e:\UA_Little_Rock\Labs\.venv\Lib\site-packages\qiskit\result\__init__.py)

In [ ]:
best_key = max(final_distribution, key=final_distribution.get)
best_bits = tuple(int(b) for b in best_key[::-1])
print("\nMost likely measured key:", best_key)
print("As q0,q1,... bits:", best_bits)
print("\nDecoded grid:")
for row in decode_assignment(best_bits):
    print(row)


Most likely measured key: 101000100
As q0,q1,... bits: (0, 0, 1, 0, 0, 0, 1, 0, 1)

Decoded grid:
(2, 0, 1)
(0, 1, 2)
(2, 1, 0)


## Why this circuit uses fewer two-qubit gates

A direct encoding for a 3×3 Sudoku grid commonly uses two qubits per cell, or 18 data qubits for nine ternary values, and then needs reversible checks for invalid encodings, row uniqueness, column uniqueness, and fixed clues.

This notebook removes most of that work classically before circuit construction:

- each row is encoded only as a valid permutation of `[0, 1, 2]`;
- fixed clues reduce the number of legal row choices;
- the oracle checks only the remaining column constraint;
- transpilation is done against a coupling map or real backend target
- simulation and hardware execution are separated so simulator basis-gate errors do not occur.

For real backends, run the hardware section with `RUN_ON_REAL_BACKEND = True`, then apply the post-selection and measurement-mitigation helpers to the returned counts.

## 2. IonQ noisless simulation

In [11]:
from qiskit_ionq import IonQProvider
import os
from dotenv import load_dotenv


provider = IonQProvider(os.environ.get("IONQ_KEY"))
simulator_backend = provider.get_backend("simulator")

# IonQ QIS-supported basis gates
ionq_basis = [
    "x", "y", "z", "h", "s", "sdg", "t", "tdg",
    "rx", "ry", "rz",
    "cx", "swap",
    "measure"
]

grover_ionq = transpile(
    grover,
    basis_gates=ionq_basis,
    optimization_level=3
)

print(grover_ionq.count_ops())
print(grover_ionq.depth())

job = simulator_backend.run(grover_ionq, shots=2048)
result = job.result()
counts_ionq = result.get_counts()

print(counts_ionq)


e:\UA_Little_Rock\Labs\.venv\Lib\site-packages\qiskit_ionq\ionq_backend.py:129: IonQTranspileLevelWarning: Transpiler default optimization_level=2. IonQ (QIS) recommends 0-1 to avoid aggressive re-synthesis; use transpile(..., optimization_level=1).
  warn_bad_transpile_level()


OrderedDict({'cx': 137, 'tdg': 67, 't': 60, 'ry': 43, 'h': 38, 'rz': 36, 'rx': 14, 'measure': 5})
308
{'00101': 2048}


In [12]:
best_key_ionq = max(counts_ionq, key=counts_ionq.get)
best_bits = tuple(int(b) for b in best_key_ionq[::-1])
print('\nMost likely key:', best_key)
print('As q0,q1,... bits:', best_bits)
print('Decoded solution:')
for row in decode_assignment(best_bits):
    print(row)


Most likely key: 00101
As q0,q1,... bits: (1, 0, 1, 0, 0)
Decoded solution:
(2, 1, 0)
(1, 0, 2)
(0, 2, 1)


## 3. IQM FakeBackend with noise simulation

In [16]:
from iqm.qiskit_iqm.fake_backends import IQMFakeApollo

backend = IQMFakeApollo()

tqc = transpile(grover, backend=backend, optimization_level=3)

job = backend.run(tqc, shots=1024)
result_iqm = job.result()
counts_iqm = result_iqm.get_counts()

print(counts_iqm)

{'11010': 40, '11101': 32, '01110': 37, '10100': 35, '01001': 22, '00100': 30, '10101': 40, '10110': 32, '00110': 26, '11100': 22, '01100': 35, '00011': 30, '10111': 30, '11111': 40, '10000': 26, '01010': 26, '11001': 32, '00001': 29, '01101': 35, '11110': 26, '00111': 32, '01011': 26, '10001': 23, '10010': 34, '00000': 46, '11011': 32, '01000': 31, '00010': 27, '01111': 38, '00101': 53, '11000': 24, '10011': 33}


In [17]:
best_key_iqm = max(counts_iqm, key=counts_iqm.get)
best_bits = tuple(int(b) for b in best_key_iqm[::-1])
print('\nMost likely key:', best_key)
print('As q0,q1,... bits:', best_bits)
print('Decoded solution:')
for row in decode_assignment(best_bits):
    print(row)


Most likely key: 00101
As q0,q1,... bits: (1, 0, 1, 0, 0)
Decoded solution:
(2, 1, 0)
(1, 0, 2)
(0, 2, 1)


### Comparison of different QPU's transpiler metrics

1. IBM and IonQ contains local noiseless simulators whereas IQM only possess FakeBackends which have noise models. 

In [18]:
comparison = [
    metrics(simulation_circuit, "Local Aer"),
    metrics(grover_ionq, "IonQ Simulator"),
    metrics(tqc, "IQM Simulator"), #(Use qiskit 1.2.4)
    #metrics(real_circuit, "IBM Fakebackend") #(Use qiskit 2.2)
]

import pandas as pd
pd.DataFrame(comparison)

,circuit,qubits,total_depth_no_measure,two_qubit_depth,ops_no_measure
0,Local Aer,7,383,137,"{'rz': 245, 'cx': 137, 'sx': 116}"
1,IonQ Simulator,7,308,137,"{'cx': 137, 'tdg': 67, 't': 60, 'ry': 43, 'h':..."
2,IQM Simulator,20,314,157,"{'r': 288, 'cz': 178}"


#### When choosing empty at the start of the game, the sudoku solutions for the fake backends gives only noise. As the IQM backend uses a fake backend, almost all of the difficulty levels just yields noise

#### For qiskit, error suppression and mitigation techniques must be applied in order to visualize the sudoku solutions in real backends instead of fake backends. I exhausted my IBM QPU time so could not perform it